#### Simple Gen AI APP Using Langchain

In [40]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [41]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

In [42]:
loader=WebBaseLoader("https://docs.langchain.com/langsmith/usage-and-billing")
loader

In [43]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/usage-and-billing', 'title': 'Usage and billing - Docs by LangChain', 'description': 'Understand LangSmith trace data retention tiers, pricing, rate limits, and usage limits.', 'language': 'en'}, page_content="Usage and billing - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationConfiguration & troubleshootingUsage and billingOverviewTraceDebugObserveReferenceQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and billingAdvanced tracing techniquesData 

In [44]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [45]:
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/usage-and-billing', 'title': 'Usage and billing - Docs by LangChain', 'description': 'Understand LangSmith trace data retention tiers, pricing, rate limits, and usage limits.', 'language': 'en'}, page_content="Usage and billing - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationConfiguration & troubleshootingUsage and billingOverviewTraceDebugObserveReferenceQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and billingAdvanced tracing techniquesData 

In [46]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [47]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [48]:
vectorstoredb

In [49]:
## Query From a vector db
query="LangSmith has two usage limits: total traces and extended"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'All traces limit\nExtended data retention traces limit\n\nThese let you limit the number of total traces, and extended data retention traces respectively.\n\u200bProperties of usage limiting\nUsage limiting is approximate, meaning that we do not guarantee the exactness of the limit. In rare cases, there may be a small period of time where additional traces are processed above the limit threshold before usage limiting begins to apply.\n\u200bSide effects of extended data retention traces limit\nThe extended data retention traces limit has side effects. If the limit is already reached, LangSmith blocks actions that would create another extended-retention trace. For example, you can no longer:\n\nrun automation rules that extend trace retention\nrun evaluators that extend trace retention'

In [50]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [51]:
## Retrieval Chain, Document chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': Fal

In [59]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

'LangSmith has two usage limits: total traces and extended traces. These are the two metrics tracked on the usage graph.'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [63]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [64]:
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [65]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x125d3c100>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            | ChatOpe

In [66]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"LangSmith has two usage limits: total traces and extended"})
response['answer']

'Based on the provided context, if the extended data retention traces limit is reached, LangSmith blocks actions that would create another extended-retention trace. Specifically, you can no longer run automation rules or evaluators that extend trace retention. Additionally, retries may not be effective if endpoints are saturated for extended periods, potentially leading to a backlog and exhaustion of retries. In such cases, contacting LangSmith Support is recommended to discuss application throughput needs and potential solutions.'

In [57]:

response

{'input': 'LangSmith has two usage limits: total traces and extended',
 'context': [Document(id='46607bfe-016c-4a86-8509-22e228015562', metadata={'source': 'https://docs.langchain.com/langsmith/usage-and-billing', 'title': 'Usage and billing - Docs by LangChain', 'description': 'Understand LangSmith trace data retention tiers, pricing, rate limits, and usage limits.', 'language': 'en'}, page_content='All traces limit\nExtended data retention traces limit\n\nThese let you limit the number of total traces, and extended data retention traces respectively.\n\u200bProperties of usage limiting\nUsage limiting is approximate, meaning that we do not guarantee the exactness of the limit. In rare cases, there may be a small period of time where additional traces are processed above the limit threshold before usage limiting begins to apply.\n\u200bSide effects of extended data retention traces limit\nThe extended data retention traces limit has side effects. If the limit is already reached, LangS

In [67]:
response['context']

[Document(id='46607bfe-016c-4a86-8509-22e228015562', metadata={'source': 'https://docs.langchain.com/langsmith/usage-and-billing', 'title': 'Usage and billing - Docs by LangChain', 'description': 'Understand LangSmith trace data retention tiers, pricing, rate limits, and usage limits.', 'language': 'en'}, page_content='All traces limit\nExtended data retention traces limit\n\nThese let you limit the number of total traces, and extended data retention traces respectively.\n\u200bProperties of usage limiting\nUsage limiting is approximate, meaning that we do not guarantee the exactness of the limit. In rare cases, there may be a small period of time where additional traces are processed above the limit threshold before usage limiting begins to apply.\n\u200bSide effects of extended data retention traces limit\nThe extended data retention traces limit has side effects. If the limit is already reached, LangSmith blocks actions that would create another extended-retention trace. For example